# RetailMind AI - Day 10: Seq2Seq + Attention Demand Forecasting

This increment extends Day 9 from one-step forecasting to a **7-day multi-horizon forecast** using an encoder-decoder LSTM with attention.

It reuses the cleaned daily demand series saved by Day 9, so no additional pandas processing is required in this notebook. Today's implementation uses only TensorFlow / Keras plus Python's standard library.

## 1. Imports and project paths

In [1]:
from pathlib import Path
import csv
import json
import tensorflow as tf
from tensorflow import keras

tf.keras.utils.set_random_seed(42)
current = Path.cwd().resolve()
PROJECT_ROOT = current.parent if current.name == 'notebooks' else current
MODELS_DIR = PROJECT_ROOT / 'models'
DAY9_DIR = MODELS_DIR / 'day9_forecast_outputs'
OUTPUT_DIR = MODELS_DIR / 'day10_forecast_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DAY9_SERIES = DAY9_DIR / 'daily_demand_series.csv'
assert DAY9_SERIES.exists(), f'Missing Day 9 series: {DAY9_SERIES}'
print('Project root:', PROJECT_ROOT)
print('Day 9 series:', DAY9_SERIES)
print('TensorFlow:', tf.__version__)


I0000 00:00:1794763529.634523 2710264 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1794763531.096632 2710264 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794763535.459892 2710264 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Project root: /home/jovyan/work/RetailMind
Day 9 series: /home/jovyan/work/RetailMind/models/day9_forecast_outputs/daily_demand_series.csv
TensorFlow: 2.21.0


## 2. Load the Day 9 daily demand series

Day 9 already converted the Kaggle source data into a clean daily series. Day 10 continues from that persisted artifact.

In [2]:
dates = []
sales = []
with open(DAY9_SERIES, newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_value = row.get('date') or row.get('') or next(iter(row.values()))
        sales_value = row.get('sales')
        if sales_value is None:
            sales_value = list(row.values())[-1]
        dates.append(date_value)
        sales.append(float(sales_value))
sales_tensor = tf.constant(sales, dtype=tf.float32)
print('Daily observations:', len(sales))
print('First date:', dates[0])
print('Last date:', dates[-1])
print('First 5 sales:', sales[:5])


Daily observations: 1826
First date: 2013-01-01
Last date: 2017-12-31
First 5 sales: [13.0, 11.0, 14.0, 13.0, 10.0]


E0000 00:00:1794763539.634744 2710264 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 3. Configure multi-horizon forecasting

The encoder reads the previous **30 days** and the decoder forecasts the next **7 days**.

In [3]:
LOOKBACK = 30
HORIZON = 7
HIDDEN_UNITS = 32
split_index = int(len(sales_tensor) * 0.80)
train_raw = sales_tensor[:split_index]
train_mean = tf.reduce_mean(train_raw)
train_std = tf.math.reduce_std(train_raw)
train_std = tf.where(train_std == 0, tf.constant(1.0), train_std)
scaled = (sales_tensor - train_mean) / train_std
print('Look-back:', LOOKBACK, 'days')
print('Forecast horizon:', HORIZON, 'days')
print('Training end index:', split_index)


Look-back: 30 days
Forecast horizon: 7 days
Training end index: 1460


## 4. Create Seq2Seq windows

Each sample contains a 30-day encoder history and a 7-day future target. The most recent 20% of target periods are held out for testing.

In [4]:
frames = tf.signal.frame(scaled, frame_length=LOOKBACK + HORIZON, frame_step=1)
X_all = frames[:, :LOOKBACK][..., tf.newaxis]
y_all = frames[:, LOOKBACK:]
target_start_indices = tf.range(LOOKBACK, LOOKBACK + tf.shape(frames)[0])
train_mask = target_start_indices + HORIZON <= split_index
test_mask = target_start_indices >= split_index
X_train = tf.boolean_mask(X_all, train_mask)
y_train = tf.boolean_mask(y_all, train_mask)
X_test = tf.boolean_mask(X_all, test_mask)
y_test = tf.boolean_mask(y_all, test_mask)
test_target_starts = tf.boolean_mask(target_start_indices, test_mask)
print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_test:', X_test.shape, 'y_test:', y_test.shape)


X_train: (1424, 30, 1) y_train: (1424, 7)
X_test: (360, 30, 1) y_test: (360, 7)


## 5. Build encoder-decoder LSTM with attention

The encoder returns one hidden state for every history day. The decoder repeats the encoder context for seven future positions. Keras `AdditiveAttention` lets each decoder step focus on the most relevant encoder days.

In [5]:
encoder_inputs = keras.Input(shape=(LOOKBACK, 1), name='history')
encoder_lstm = keras.layers.LSTM(HIDDEN_UNITS, return_sequences=True, return_state=True, name='encoder_lstm')
encoder_seq, encoder_h, encoder_c = encoder_lstm(encoder_inputs)

decoder_seed = keras.layers.RepeatVector(HORIZON, name='repeat_context')(encoder_h)
decoder_lstm = keras.layers.LSTM(HIDDEN_UNITS, return_sequences=True, name='decoder_lstm')
decoder_seq = decoder_lstm(decoder_seed, initial_state=[encoder_h, encoder_c])

attention = keras.layers.AdditiveAttention(name='demand_attention')
attention_context, attention_scores = attention(
    [decoder_seq, encoder_seq],
    return_attention_scores=True
)
combined = keras.layers.Concatenate(name='decoder_plus_attention')([decoder_seq, attention_context])
dense_hidden = keras.layers.TimeDistributed(keras.layers.Dense(16, activation='relu'))(combined)
forecast_seq = keras.layers.TimeDistributed(keras.layers.Dense(1), name='forecast')(dense_hidden)
forecast = keras.layers.Reshape((HORIZON,), name='multi_horizon_forecast')(forecast_seq)

seq2seq_model = keras.Model(encoder_inputs, forecast, name='retailmind_seq2seq_attention')
seq2seq_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
seq2seq_model.summary()


Model: "retailmind_seq2seq_attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ history             │ (None, 30, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 30, 32),  │      4,352 │ history[0][0]     │
│                     │ (None, 32),       │            │                   │
│                     │ (None, 32)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_context      │ (None, 7, 32)     │          0 │ encoder_lstm[0][… │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ (None, 7, 32)     │      8,320 │ repeat_context[0… │
│                     │                   │            │ encoder_lstm[0][… │
│                     │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ demand_attention    │ [(None, 7, 32),   │         32 │ decoder_lstm[0][… │
│ (AdditiveAttention) │ (None, 7, 30)]    │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_plus_atten… │ (None, 7, 64)     │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ demand_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_distributed    │ (None, 7, 16)     │      1,040 │ decoder_plus_att… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ forecast            │ (None, 7, 1)      │         17 │ time_distributed… │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_horizon_fore… │ (None, 7)         │          0 │ forecast[0][0]    │
│ (Reshape)           │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 13,761 (53.75 KB)

 Trainable params: 13,761 (53.75 KB)

 Non-trainable params: 0 (0.00 B)

## 6. Train with chronological validation and EarlyStopping

In [6]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)
history = seq2seq_model.fit(
    X_train, y_train,
    epochs=40,
    batch_size=32,
    validation_split=0.20,
    shuffle=False,
    callbacks=[early_stop],
    verbose=1
)
print('Epochs run:', len(history.history['loss']))
print('Best validation loss:', round(min(history.history['val_loss']), 4))


Epoch 1/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 4s 26ms/step - loss: 0.7524 - mae: 0.6840 - val_loss: 0.8440 - val_mae: 0.7290
Epoch 2/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.6410 - mae: 0.6279 - val_loss: 0.7486 - val_mae: 0.6936
Epoch 3/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 0.6375 - mae: 0.6258 - val_loss: 0.7462 - val_mae: 0.6921
Epoch 4/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.6352 - mae: 0.6245 - val_loss: 0.7430 - val_mae: 0.6912
Epoch 5/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.6337 - mae: 0.6237 - val_loss: 0.7409 - val_mae: 0.6904
Epoch 6/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.6329 - mae: 0.6232 - val_loss: 0.7391 - val_mae: 0.6899
Epoch 7/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.6319 - mae: 0.6228 - val_loss: 0.7378 - val_mae: 0.6896
Epoch 8/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.6310 - mae: 0.6224 - val_loss: 0.7372 - val_mae: 0.6891
Epoch 9/40
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.630

## 7. Evaluate all seven forecast horizons

In [7]:
pred_scaled = seq2seq_model.predict(X_test, verbose=0)
predicted = tf.maximum(pred_scaled * train_std + train_mean, 0.0)
actual = y_test * train_std + train_mean
errors = predicted - actual
overall_mae = tf.reduce_mean(tf.abs(errors))
overall_rmse = tf.sqrt(tf.reduce_mean(tf.square(errors)))
horizon_mae = tf.reduce_mean(tf.abs(errors), axis=0)
horizon_rmse = tf.sqrt(tf.reduce_mean(tf.square(errors), axis=0))
print('Overall 7-day MAE:', round(float(overall_mae), 4))
print('Overall 7-day RMSE:', round(float(overall_rmse), 4))
print('\nPer-horizon metrics:')
for h in range(HORIZON):
    print(f'Day +{h+1}: MAE={float(horizon_mae[h]):.4f}, RMSE={float(horizon_rmse[h]):.4f}')


Overall 7-day MAE: 4.5754
Overall 7-day RMSE: 5.8059

Per-horizon metrics:
Day +1: MAE=4.5988, RMSE=5.8311
Day +2: MAE=4.5780, RMSE=5.8157
Day +3: MAE=4.6039, RMSE=5.8400
Day +4: MAE=4.6164, RMSE=5.8483
Day +5: MAE=4.5476, RMSE=5.7909
Day +6: MAE=4.5383, RMSE=5.7508
Day +7: MAE=4.5450, RMSE=5.7640


## 8. Compare with the Day 9 single-step LSTM

Day 9's MAE/RMSE are loaded from its saved metrics. The Day +1 metrics from the Seq2Seq model provide the closest like-for-like comparison.

In [8]:
DAY9_METRICS = MODELS_DIR / 'day9_demand_metrics.json'
day9_metrics = {}
if DAY9_METRICS.exists():
    with open(DAY9_METRICS) as f:
        day9_metrics = json.load(f)
    print('Day 9 single-step MAE:', round(day9_metrics['mae'], 4))
    print('Day 9 single-step RMSE:', round(day9_metrics['rmse'], 4))
else:
    print('Day 9 metrics file was not found; continuing with Day 10 metrics only.')
print('Day 10 Day+1 MAE:', round(float(horizon_mae[0]), 4))
print('Day 10 Day+1 RMSE:', round(float(horizon_rmse[0]), 4))


Day 9 single-step MAE: 4.5853
Day 9 single-step RMSE: 5.8225
Day 10 Day+1 MAE: 4.5988
Day 10 Day+1 RMSE: 5.8311


## 9. Build an attention-inspection model

The attention scores have shape `(samples, 7 future steps, 30 history days)`. Larger weights indicate history positions receiving more attention for a selected forecast horizon.

In [9]:
attention_model = keras.Model(encoder_inputs, [forecast, attention_scores])
example_forecast_scaled, example_attention = attention_model.predict(X_test[-1:], verbose=0)
example_forecast = tf.maximum(example_forecast_scaled[0] * train_std + train_mean, 0.0)
weights = example_attention[0]
print('Attention tensor:', weights.shape)
print('7-day example forecast:', [round(float(v), 2) for v in example_forecast])


Attention tensor: (7, 30)
7-day example forecast: [15.81, 15.57, 15.44, 15.39, 15.39, 15.41, 15.44]


## 10. Visualize which history days drive each forecast

To stay within the day's TensorFlow/Keras-only constraint, attention is displayed as ranked weights rather than using an additional plotting library. For each future day, the five most-attended history lags are shown. Lag 1 means yesterday; lag 30 means 30 days ago.

In [10]:
attention_rows = []
for future_step in range(HORIZON):
    step_weights = weights[future_step]
    top = tf.math.top_k(step_weights, k=5)
    row = []
    for index, weight in zip(top.indices.numpy(), top.values.numpy()):
        lag = LOOKBACK - int(index)
        row.append((lag, float(weight)))
    attention_rows.append(row)
    print(f'Forecast Day +{future_step+1}:')
    for lag, weight in row:
        bar = '█' * max(1, int(weight * 100))
        print(f'  lag {lag:2d} days | weight={weight:.4f} | {bar}')


Forecast Day +1:
  lag  3 days | weight=0.0415 | ████
  lag  2 days | weight=0.0408 | ████
  lag  1 days | weight=0.0398 | ███
  lag  4 days | weight=0.0384 | ███
  lag  5 days | weight=0.0370 | ███
Forecast Day +2:
  lag  3 days | weight=0.0416 | ████
  lag  2 days | weight=0.0409 | ████
  lag  1 days | weight=0.0399 | ███
  lag  4 days | weight=0.0385 | ███
  lag  5 days | weight=0.0370 | ███
Forecast Day +3:
  lag  3 days | weight=0.0417 | ████
  lag  2 days | weight=0.0410 | ████
  lag  1 days | weight=0.0400 | ████
  lag  4 days | weight=0.0385 | ███
  lag  5 days | weight=0.0371 | ███
Forecast Day +4:
  lag  3 days | weight=0.0417 | ████
  lag  2 days | weight=0.0411 | ████
  lag  1 days | weight=0.0401 | ████
  lag  4 days | weight=0.0386 | ███
  lag  5 days | weight=0.0371 | ███
Forecast Day +5:
  lag  3 days | weight=0.0417 | ████
  lag  2 days | weight=0.0411 | ████
  lag  1 days | weight=0.0401 | ████
  lag  4 days | weight=0.0386 | ███
  lag  5 days | weight=0.0371 | ███
Fo

## 11. Save forecast and attention outputs

In [11]:
MODEL_PATH = MODELS_DIR / 'demand_seq2seq_attention.keras'
FORECAST_PATH = OUTPUT_DIR / 'multi_horizon_forecast.csv'
ATTENTION_PATH = OUTPUT_DIR / 'attention_weights.csv'
METRICS_PATH = MODELS_DIR / 'day10_demand_metrics.json'
seq2seq_model.save(MODEL_PATH)

with open(FORECAST_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    header = ['sample_index'] + [f'actual_day_{i}' for i in range(1, HORIZON+1)] + [f'predicted_day_{i}' for i in range(1, HORIZON+1)]
    writer.writerow(header)
    for i in range(len(predicted)):
        writer.writerow([i] + [float(v) for v in actual[i].numpy()] + [float(v) for v in predicted[i].numpy()])

with open(ATTENTION_PATH, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['forecast_horizon', 'history_lag_days', 'attention_weight'])
    for future_step, row in enumerate(attention_rows, start=1):
        for lag, weight in row:
            writer.writerow([future_step, lag, weight])

metrics = {
    'lookback_days': LOOKBACK,
    'forecast_horizon_days': HORIZON,
    'epochs_run': len(history.history['loss']),
    'best_validation_loss': float(min(history.history['val_loss'])),
    'overall_mae': float(overall_mae),
    'overall_rmse': float(overall_rmse),
    'day1_mae': float(horizon_mae[0]),
    'day1_rmse': float(horizon_rmse[0]),
    'per_horizon_mae': [float(v) for v in horizon_mae],
    'per_horizon_rmse': [float(v) for v in horizon_rmse],
    'day9_mae': day9_metrics.get('mae'),
    'day9_rmse': day9_metrics.get('rmse'),
    'model': 'models/demand_seq2seq_attention.keras',
    'forecast_file': 'models/day10_forecast_outputs/multi_horizon_forecast.csv',
    'attention_file': 'models/day10_forecast_outputs/attention_weights.csv'
}
with open(METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved model:', MODEL_PATH)
print('Saved forecast:', FORECAST_PATH)
print('Saved attention weights:', ATTENTION_PATH)
print('Saved metrics:', METRICS_PATH)


Saved model: /home/jovyan/work/RetailMind/models/demand_seq2seq_attention.keras
Saved forecast: /home/jovyan/work/RetailMind/models/day10_forecast_outputs/multi_horizon_forecast.csv
Saved attention weights: /home/jovyan/work/RetailMind/models/day10_forecast_outputs/attention_weights.csv
Saved metrics: /home/jovyan/work/RetailMind/models/day10_demand_metrics.json


## 12. Upgraded multi-week demand interface

`forecast_multi_horizon()` accepts the latest 30 daily-sales values and emits the next seven demand forecasts for procurement planning.

In [12]:
def forecast_multi_horizon(recent_sales):
    if len(recent_sales) != LOOKBACK:
        raise ValueError(f'Provide exactly {LOOKBACK} recent daily sales values.')
    recent = tf.constant(recent_sales, dtype=tf.float32)
    recent_scaled = (recent - train_mean) / train_std
    model_x = tf.reshape(recent_scaled, (1, LOOKBACK, 1))
    pred = seq2seq_model.predict(model_x, verbose=0)[0]
    pred = tf.maximum(pred * train_std + train_mean, 0.0)
    return [round(float(v), 2) for v in pred]
latest_30 = sales[-LOOKBACK:]
next_7_days = forecast_multi_horizon(latest_30)
print('Next 7-day demand forecast:', next_7_days)


Next 7-day demand forecast: [16.86, 17.05, 17.28, 17.51, 17.74, 17.95, 18.13]


## 13. Verify Day 10 artifacts

In [13]:
required = [MODEL_PATH, FORECAST_PATH, ATTENTION_PATH, METRICS_PATH]
for path in required:
    print(('OK' if path.exists() else 'MISSING'), '-', path.relative_to(PROJECT_ROOT))
assert all(path.exists() for path in required), 'A Day 10 artifact is missing.'
print('Day 10 completed successfully.')


OK - models/demand_seq2seq_attention.keras
OK - models/day10_forecast_outputs/multi_horizon_forecast.csv
OK - models/day10_forecast_outputs/attention_weights.csv
OK - models/day10_demand_metrics.json
Day 10 completed successfully.


## 14. README-ready summary

In [14]:
print('DAY 10 - SEQ2SEQ + ATTENTION FORECASTING')
print('-' * 65)
print('Look-back:', LOOKBACK, 'days')
print('Forecast horizon:', HORIZON, 'days')
print('Epochs run:', len(history.history['loss']))
print('Overall MAE:', f'{float(overall_mae):.4f}')
print('Overall RMSE:', f'{float(overall_rmse):.4f}')
print('Day +1 MAE:', f'{float(horizon_mae[0]):.4f}')
print('Day +1 RMSE:', f'{float(horizon_rmse[0]):.4f}')
if day9_metrics:
    print('Day 9 MAE:', f"{day9_metrics['mae']:.4f}")
    print('Day 9 RMSE:', f"{day9_metrics['rmse']:.4f}")
print('Next 7-day forecast:', next_7_days)
print('Model: models/demand_seq2seq_attention.keras')
print('Attention file: models/day10_forecast_outputs/attention_weights.csv')


DAY 10 - SEQ2SEQ + ATTENTION FORECASTING
-----------------------------------------------------------------
Look-back: 30 days
Forecast horizon: 7 days
Epochs run: 18
Overall MAE: 4.5754
Overall RMSE: 5.8059
Day +1 MAE: 4.5988
Day +1 RMSE: 5.8311
Day 9 MAE: 4.5853
Day 9 RMSE: 5.8225
Next 7-day forecast: [16.86, 17.05, 17.28, 17.51, 17.74, 17.95, 18.13]
Model: models/demand_seq2seq_attention.keras
Attention file: models/day10_forecast_outputs/attention_weights.csv


## Requirement coverage

- Reshaped Day 9 demand history into encoder-history / multi-step-target windows
- Built an LSTM encoder-decoder Seq2Seq model
- Added an attention mechanism over the 30-day encoder history
- Generated seven future demand steps in one forecast
- Compared Day +1 accuracy with the Day 9 single-step LSTM
- Displayed ranked attention weights showing which history lags influence each horizon
- Upgraded the demand interface to emit a seven-day forecast for procurement planning
- Saved model, forecasts, attention weights, and metrics

### Business value
A seven-day demand outlook gives procurement teams more planning lead time than the Day 9 single-step forecast. Attention also adds interpretability by exposing which historical demand periods influence each future step.

### Open blocker
The current Phase 3 prototype still models one store/item demand series and does not include promotions, holidays, pricing, weather, or other external demand drivers.